# Calibrate an objective pair

Measure how far apart a reference objective and a target objective look and focus, so the driver can keep the same spot in view when the objective changes. Do the limits and the orientation first.

Every Measure cell is safe to run again. It first checks that the expected objective is in place. If it is not, the cell refuses without losing anything, so you can switch back and simply run it again. Once the check passes, the cell replaces its own previous measurement and anything that was worked out from it.

## Configure

Edit the three values below for your own run, then run the cell:

- `acquisition_name` names the objective pair you are about to measure, for example `"10x-60x"`. Each pair gets its own folder inside the session.
- `session_id` names the calibration session that collects all pairs measured in one sitting. A date and a short label work well.
- `reference_slot` is the nosepiece slot of the reference objective, counted from 0: the objective every other objective is measured against. Keep it the same for every pair in one session.

In FLUOVIEW, put in the reference objective and set up the imaging you will use. Between the steps below you change only the objective, the zoom and the z-stack, as each step says.

In [ ]:
import bootstrap

from zmart_drivers.evident.fv4000.rdk.driver_setup.optical_calibration import (
    adopt_calibration as adopt,
)
from zmart_drivers.evident.fv4000.rdk.driver_setup.optical_calibration import (
    calibrate_objective_pair as objective_pair,
)

session = objective_pair.start_session(
    acquisition_name="10x-60x",
    session_id="2026-10-29_calibration_1",
    reference_slot=0,
)

## Measure 1: reference focus

With the reference objective, focus on the sample, and set up a z-stack in FLUOVIEW (start and end, for example 80 µm in 1 µm steps) that brackets the sharp plane comfortably. Choose a zoom whose pixel size the target objective can match.

The cell runs that stack and finds its sharpest slice. The sharp plane must fall inside the stack, away from its ends, or the cell refuses and asks you to refocus and run it again.

In [ ]:
objective_pair.measure_parfocality_reference(session);

## Measure 2: target focus

Switch only to the target objective, and set the zoom so its pixel size matches the reference's. Keep the same z-stack and change nothing else.

This cell decides which objective counts as the target. If you ran it with the wrong objective, switch to the intended one and run it again: the wrong measurement is discarded and the pair is redefined.

In [ ]:
objective_pair.measure_parfocality_target(session);

## Measure 3: reference X/Y

Switch back to the reference objective, with its zoom from Measure 1, and switch the z-stack off. The cell brings the stage back to where Measure 1 began, sets the focus to the sharpest plane it found, and takes the reference picture.

The X/Y offset is found by matching the reference and target pictures against each other. That only means something when both have exactly the same pixel size: otherwise the same feature lands a different number of pixels away in each.

In [ ]:
objective_pair.measure_parcentricity_reference(session);

## Measure 4: target X/Y

Switch only to the target objective, with its zoom from Measure 2. Do not move the stage.

If the summary reports WEAK VOTE, the matching methods could not agree on the shift and nothing was saved. That usually means the two pictures do not share enough detail. Move to a spot with more structure under the reference objective, refocus, and run Measure 3 and Measure 4 again.

Use the alignment pictures below to confirm by eye that the reference and the corrected target overlap before you save and adopt.

In [ ]:
summary = objective_pair.measure_parcentricity_target_and_save(session)

## Save and adopt

This saves the completed notebook, checks that the X/Y measurement is there, and then publishes the calibration, so the driver uses it from the next connection.

To calibrate another pair (for example 10x-20x), go back to Configure, change only `acquisition_name`, and run the notebook again from there with the same reference objective. Pairs that share the reference combine by themselves: once 10x-60x and 10x-20x are measured, the 20x-60x offset follows from them without a measurement of its own. Adopting again after each pair is fine: adoption always publishes everything measured in the session so far.

In [ ]:
# Save and Adopt
adopted = bootstrap.NOTEBOOK.save_and_adopt(
    lambda notebook: adopt.adopt_calibration(session, notebook_paths=[notebook])
)
print(f"Calibration adopted: {adopted['calibration_path']}")